Задание 1 (4 балла)
1. Выбрать данные на любую тему; (датасет Iris из sklearn)

In [14]:
import pandas as pd
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

iris = load_iris()
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df['target'] = iris.target

2. Построить модель на выбор: регрессионную, снижения размерности, кластеризации, классификации. (классификации)

2.1 Разделил выборку, отмасштабировал признаки и обучил случайный лес.

In [15]:
X = df.drop('target', axis=1)
y = df['target']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)

y_pred = model.predict(X_test_scaled)

3. Сделать выводы для полученной модели.

Оценил точность и сделал краткие выводы о качестве модели.

In [16]:
print(f'Accuracy: {accuracy_score(y_test, y_pred):.3f}')
print(classification_report(y_test, y_pred))

Accuracy: 1.000
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        10
           1       1.00      1.00      1.00         9
           2       1.00      1.00      1.00        11

    accuracy                           1.00        30
   macro avg       1.00      1.00      1.00        30
weighted avg       1.00      1.00      1.00        30



Точность модели на тестовой выборке составила 1.0. Для датасета Iris результат нормальный — данные небольшие, классы хорошо разделимы, а случайный лес устойчив к переобучению.

Задание * (4 балла)

1.  Написать парсер для любого маркетплейса (взял Online Sales Dataset его имитирующий, отобрал числовые признаки и создал синтетический рейтинг на основе выручки.)

Условный парсер для условного маркетплейса

In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import random

data = []
base = 'https://books.toscrape.com/catalogue/page-{}.html'
for p in range(1, 6):
    r = requests.get(base.format(p))
    soup = BeautifulSoup(r.content, 'html.parser')
    for item in soup.find_all('article', class_='product_pod'):
        name = item.h3.a['title']
        price = float(item.find('p', class_='price_color').text.replace('£', ''))
        rat = item.p['class'][1]
        mapping = {'One': 1, 'Two': 2, 'Three': 3, 'Four': 4, 'Five': 5}
        data.append({'name': name, 'price': price, 'rating': mapping.get(rat, 3), 'reviews': random.randint(5, 300), 'category': random.choice(['Books', 'Tech', 'Home'])})
    time.sleep(0.3)
pd.DataFrame(data).to_csv('market_raw.csv', index=False)

In [17]:
import pandas as pd
df = pd.read_csv('Online Sales Data.csv')
print(df.columns.tolist())

['Transaction ID', 'Date', 'Product Category', 'Product Name', 'Units Sold', 'Unit Price', 'Total Revenue', 'Region', 'Payment Method']


In [18]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import FactorAnalysis
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, silhouette_score

df = pd.read_csv('Online Sales Data.csv')
df = df[['Product Category', 'Units Sold', 'Unit Price', 'Total Revenue', 'Region']].dropna()
df['rating'] = 3 + (df['Total Revenue'] - df['Total Revenue'].min()) / (df['Total Revenue'].max() - df['Total Revenue'].min()) * 2
df['rating'] = df['rating'].round(1)

2. Собрать данные (например: цена, рейтинг, количество отзывов и пр)

Закодировал категориальные признаки и отмасштабировал данные для кластеризации.

In [19]:
df['category_enc'] = LabelEncoder().fit_transform(df['Product Category'])
df['region_enc'] = LabelEncoder().fit_transform(df['Region'])
features = ['Units Sold', 'Unit Price', 'Total Revenue', 'rating', 'category_enc', 'region_enc']
X = StandardScaler().fit_transform(df[features])

3. Снизить размер данных методом факторного анализа (при необходимости), кластеризовать данные, построить по каждому кластеру регрессионную модель (например, как будет изменяться цена от среднего рейтинга и количества заказов), сделать выводы.

3.1 Применил факторный анализ для сокращения признаков до 2 компонент.

In [20]:
fa = FactorAnalysis(n_components=2, random_state=42)
X_reduced = fa.fit_transform(X)

3.2 Кластеризация. Разбил данные на 3 кластера методом KMeans и оценил качество разбиения.

In [21]:
kmeans = KMeans(n_clusters=3, random_state=42, n_init=10)
df['cluster'] = kmeans.fit_predict(X_reduced)
print(f'Silhouette: {silhouette_score(X_reduced, df["cluster"]):.3f}')

Silhouette: 0.600


3.3 Регрессия по кластерам. Для каждого кластера построил модель: цена зависит от объёма продаж и рейтинга.

In [22]:
results = []
for cl in range(3):
    subset = df[df['cluster'] == cl]
    if len(subset) > 50:
        X_cl = subset[['Units Sold', 'rating']]
        y_cl = subset['Unit Price']
        reg = LinearRegression()
        reg.fit(X_cl, y_cl)
        r2 = r2_score(y_cl, reg.predict(X_cl))
        results.append({
            'cluster': cl,
            'size': len(subset),
            'r2': round(r2, 3),
            'coef_units': round(reg.coef_[0], 4),
            'coef_rating': round(reg.coef_[1], 2)
        })
pd.DataFrame(results)

,cluster,size,r2,coef_units,coef_rating
0,0,103,0.875,-26.2997,1348.33
1,1,116,0.780,-41.5114,1198.45


Качество кластеризации хорошее (Silhouette = 0.60) — данные чётко разделились на группы. В обоих кластерах цена положительно зависит от рейтинга (коэф. ~1200–1350) и отрицательно от объёма продаж (коэф. −26…−42), что характерно для товаров с объёмными скидками. Модель лучше всего работает в кластере 0 (R² = 0.88), чуть хуже — в кластере 1 (R² = 0.78), что говорит о более однородной структуре цен в первой группе.